# Dust density on an adaptively refined meshProposal-quality figures of the particle-mesh dust density from AthenaK `.bin` dumps, withthe MeshBlocks of each refinement level drawn on top.Everything you are likely to want to change lives in **§2 (what to plot)** and**§4 (style)**; §3 and §5 are the machinery and should not need editing. The plottingfunction takes every style choice as a keyword argument, so you can also override any ofthem per call without touching the defaults.*Data*: any AthenaK run that writes `file_type = bin` with `variable = dust_dpm`. Thereader handles a uniform mesh and any number of AMR levels.*Reference*: `validation/figures/plot_dust6_ba_amr3.jl` produced the figures in section 12of the validation document; this notebook is the same machinery, unpacked for tweaking.

## 1. Setup

In [ ]:
using CairoMakie, Printf, Statistics
CairoMakie.activate!(type = "png")      # "svg" or "pdf" for vector output

const REPO = expanduser("~/Library/CloudStorage/Dropbox/Research/code/athenak-multigrid")
include(joinpath(REPO, "scripts", "athenak_bin.jl"))   # read_bin, block_outlines

# where the runs live (validation/run/dust6/<case>/bin/*.bin)
const DATA = joinpath(REPO, "validation", "run", "dust6")
readdir(DATA)

## 2. What to plot`CASE` is a directory under `DATA`, `BASE` the `<job>/basename` of that run, and `TSNAP`the simulation time you want (the nearest dump is used; `Inf` takes the last one).

In [ ]:
CASE  = "BA_amr4_mb8"        # BA_amr3 | BA_amr4 | BA_amr4_mb8 | AB_amr256 | ...
BASE  = "si_BA4m8"           # si_BA3 | si_BA4 | si_BA4m8 | si_AB_amr256 | ...
TSNAP = 230.0                # Inf = last dump

BINDIR = joinpath(DATA, CASE, "bin")
dumps  = sort(filter(f -> occursin("$BASE.dust_dpm.", f), readdir(BINDIR)))
times  = [read_bin(joinpath(BINDIR, f)).time for f in dumps]
@printf("%d dumps, t = %.1f to %.1f\n", length(dumps), first(times), last(times))

## 3. Loading and assembling`load_snapshot` returns the dust density on the **finest** grid in the file (coarse blocksare repeated, so a level-1 block appears as a 2x2 patch of identical cells -- thatblockiness is real resolution, not a plotting artifact), together with the cell-edgecoordinates, the per-level MeshBlock census, and the block outlines.

In [ ]:
"Nearest dump to time t, as a BinFileData."
function snapshot_at(bindir, base, t)
    fs = sort(filter(f -> occursin("$base.dust_dpm.", f), readdir(bindir)))
    isempty(fs) && error("no $base.dust_dpm.*.bin in $bindir")
    ts = [read_bin(joinpath(bindir, f)).time for f in fs]
    read_bin(joinpath(bindir, fs[isfinite(t) ? argmin(abs.(ts .- t)) : length(fs)]))
end

"""
    load_snapshot(bindir, base, t) -> NamedTuple

`rho` (dust density on the finest grid), `xe`/`ze` (cell edges), `time`, `lmax`,
`nlev` (blocks per level), `ncells`, `nuniform`, `outlines` (vector of (rect, level)).
"""
function load_snapshot(bindir, base, t)
    fd   = snapshot_at(bindir, base, t)
    lev  = fd.mb_logical[:, 4]
    lmax = maximum(lev)
    s    = 1 << lmax
    m1, m2 = fd.nx_mb[1], fd.nx_mb[2]
    rho  = fill(NaN, fd.Nx1 * s, fd.Nx2 * s)
    for m in 1:fd.n_mbs
        lx1, lx2, _, l = fd.mb_logical[m, :]
        blk = fd.mb_data["dustdpm"][m][:, :, 1]
        e = 1 << (lmax - l)
        e > 1 && (blk = repeat(blk, inner = (e, e)))
        i0, j0 = lx1 * m1 * e, lx2 * m2 * e
        rho[i0+1:i0+m1*e, j0+1:j0+m2*e] = blk
    end
    any(isnan, rho) && error("gaps in the assembled grid")
    (rho = rho,
     xe = range(fd.x1min, fd.x1max, length = size(rho, 1) + 1),
     ze = range(fd.x2min, fd.x2max, length = size(rho, 2) + 1),
     time = fd.time, lmax = lmax,
     nlev = [count(lev .== l) for l in 0:lmax],
     ncells = fd.n_mbs * m1 * m2, nuniform = (fd.Nx1 * s) * (fd.Nx2 * s),
     outlines = block_outlines(fd))
end

snap = load_snapshot(BINDIR, BASE, TSNAP)
@printf("t = %.1f   blocks/level = %s   %.1f%% of the uniform %d^2 mesh\n",
        snap.time, string(snap.nlev), 100 * snap.ncells / snap.nuniform,
        round(Int, sqrt(snap.nuniform)))

## 4. StyleDefaults for the figure. Every one is also a keyword of `plot_dust` below, so`plot_dust(snap; colormap = :magma)` overrides just that one for a single figure.* `SCALE` -- `:log` plots log10(rho_p / <rho_p>); `:linear` plots rho_p / rho_g on a  linear scale from 0, the convention of Johansen & Youdin (2007) Figs. 2 and 5  (their limit is 1 for run BA, 5 for run AB).* `LEVEL_COLORS` -- one colour per level, coarsest first. Keep them off the colormap's  own axis or the outlines vanish: against `cividis` (dark blue to yellow), magenta,  spring green and red read well; cyan and white do not.* `UNITS` -- `:H` labels axes in scale heights, `:etar` in eta*r (set `ETAR` to eta r / H).

In [ ]:
SCALE        = :log             # :log | :linear
COLORMAP     = :cividis
COLORRANGE   = (-1.7, 0.8)      # log10 rho_p/<rho_p>; for :linear use e.g. (0, 1)
EPS0         = 0.2              # <rho_p>/rho_g of the run (BA: 0.2, AB: 1.0)
LEVEL_COLORS = (:white, :magenta, :springgreen, :red)   # root, l1, l2, l3, ...
LEVEL_WIDTHS = (2.2, 1.8, 1.1, 0.7)
UNITS        = :H               # :H | :etar
ETAR         = 0.05             # eta r in units of H (eta v_K = 0.05 c_s)
FIGSIZE      = (950, 880)
FONTSIZE     = 14
SHOW_TITLE   = true
SHOW_LEGEND  = true
SHOW_MESH    = true
PX_PER_UNIT  = 1.6;             # raster upscaling for png output

## 5. The plot`plot_dust(snap; kwargs...)` returns the `Figure`. Pass `title = "..."` for your owncaption, `title = nothing` to suppress it, `mesh_levels = [2, 3]` to outline only somelevels, or `colorbar = false` to drop the bar (useful when a figure is one panel ofseveral).

In [ ]:
function plot_dust(snap;
        scale = SCALE, colormap = COLORMAP, colorrange = COLORRANGE, eps0 = EPS0,
        level_colors = LEVEL_COLORS, level_widths = LEVEL_WIDTHS,
        units = UNITS, etar = ETAR, figsize = FIGSIZE, fontsize = FONTSIZE,
        show_mesh = SHOW_MESH, mesh_levels = nothing, colorbar = true,
        legend = SHOW_LEGEND, title = :auto, floor = 1e-2)

    sc  = units === :etar ? 1 / etar : 1.0
    lab = units === :etar ? ("x / ηr", "z / ηr") : ("x  (H)", "z  (H)")

    field, cblabel = if scale === :log
        log10.(max.(snap.rho ./ eps0, floor)), "log₁₀ ρ_p / ⟨ρ_p⟩"
    else
        snap.rho, "ρ_p / ρ_g"
    end

    ttl = title === :auto ?
        @sprintf("t = %.0f Ω⁻¹:  %s MeshBlocks, %.0f%% of the uniform %d² mesh",
                 snap.time, join(string.(snap.nlev), " + "),
                 100 * snap.ncells / snap.nuniform, round(Int, sqrt(snap.nuniform))) : title

    fig = Figure(size = figsize, fontsize = fontsize)
    ax  = Axis(fig[1, 1], xlabel = lab[1], ylabel = lab[2], aspect = DataAspect(),
               title = ttl === nothing ? "" : ttl, titlesize = fontsize + 2)
    hm  = heatmap!(ax, snap.xe .* sc, snap.ze .* sc, field;
                   colormap = colormap, colorrange = colorrange)

    if show_mesh
        want = mesh_levels === nothing ? (0:snap.lmax) : mesh_levels
        for (r, l) in snap.outlines
            l in want || continue
            lines!(ax, [r[1], r[2], r[2], r[1], r[1]] .* sc,
                       [r[3], r[3], r[4], r[4], r[3]] .* sc,
                   color = (level_colors[l+1], 0.95), linewidth = level_widths[l+1])
        end
        if legend
            hs, ls = [], String[]
            for l in 0:snap.lmax
                snap.nlev[l+1] > 0 || continue
                push!(hs, LineElement(color = level_colors[l+1],
                                      linewidth = max(level_widths[l+1], 2.0)))
                push!(ls, @sprintf("level %d: %d blocks", l, snap.nlev[l+1]))
            end
            !isempty(hs) && axislegend(ax, hs, ls, position = :lb, framevisible = true,
                framecolor = (:white, 0.4), backgroundcolor = (:black, 0.55),
                labelcolor = :white, labelsize = fontsize - 2, patchsize = (26, 12))
        end
    end
    colorbar && Colorbar(fig[1, 2], hm, label = cblabel, width = 14)
    fig
end

plot_dust(snap)

## 6. SavingPNG for drafts, PDF for the proposal (vector text and outlines, raster heatmap).

In [ ]:
OUT = joinpath(homedir(), "Desktop")     # or wherever you collect proposal figures
fig = plot_dust(snap)
save(joinpath(OUT, "dust_amr_$(CASE)_t$(round(Int, snap.time)).png"), fig,
     px_per_unit = PX_PER_UNIT)
CairoMakie.activate!(type = "pdf")
save(joinpath(OUT, "dust_amr_$(CASE)_t$(round(Int, snap.time)).pdf"), plot_dust(snap))
CairoMakie.activate!(type = "png")
println("saved to ", OUT)

## 7. VariationsThree things you are likely to want. Each is a single call.**A publication panel** -- no title, no legend, no colorbar, mesh only at the finest twolevels.

In [ ]:
plot_dust(snap; title = nothing, legend = false, colorbar = false,
          mesh_levels = [snap.lmax - 1, snap.lmax], figsize = (700, 700))

**The published convention** (Johansen & Youdin 2007): linear rho_p/rho_g from 0, nomesh, axes in eta r. Their upper limit is 1 for run BA and 5 for run AB.

In [ ]:
plot_dust(snap; scale = :linear, colorrange = (0, 1), colormap = :inferno,
          units = :etar, show_mesh = false, title = @sprintf("t = %.0f Ω⁻¹", snap.time))

**A time sequence.** Edit the times; the panels share one colour scale.

In [ ]:
let ts = [40.0, 140.0, 200.0, 230.0]
    fig = Figure(size = (1100, 1050), fontsize = 13)
    for (n, t) in enumerate(ts)
        s = load_snapshot(BINDIR, BASE, t)
        r, c = divrem(n - 1, 2) .+ (1, 1)
        ax = Axis(fig[r, c], aspect = DataAspect(), title = @sprintf("t = %.0f Ω⁻¹", s.time),
                  xlabel = r == 2 ? "x  (H)" : "", ylabel = c == 1 ? "z  (H)" : "")
        heatmap!(ax, s.xe, s.ze, log10.(max.(s.rho ./ EPS0, 1e-2));
                 colormap = COLORMAP, colorrange = COLORRANGE)
        for (rect, l) in s.outlines
            l == s.lmax || continue
            lines!(ax, [rect[1], rect[2], rect[2], rect[1], rect[1]],
                       [rect[3], rect[3], rect[4], rect[4], rect[3]],
                   color = (LEVEL_COLORS[l+1], 0.9), linewidth = 0.6)
        end
    end
    Colorbar(fig[1:2, 3], colormap = COLORMAP, colorrange = COLORRANGE,
             label = "log₁₀ ρ_p / ⟨ρ_p⟩", width = 14)
    fig
end

## 8. Numbers behind the figureThe mesh census, and where each level sits in density -- useful for a caption, and thecheck that the criterion is placing blocks by density rather than being overruled by the2:1 rule (if two adjacent levels have the same median, they are being pinned, not chosen).

In [ ]:
let fd = snapshot_at(BINDIR, BASE, TSNAP)
    lev = fd.mb_logical[:, 4]
    mx  = [maximum(fd.mb_data["dustdpm"][m]) for m in 1:fd.n_mbs]
    @printf("t = %.1f,  %d blocks of %d cells\n", fd.time, fd.n_mbs, prod(fd.nx_mb))
    for l in 0:maximum(lev)
        idx = findall(lev .== l); isempty(idx) && continue
        @printf("  level %d: %4d blocks, block width %.4f H, median block-max ρ_p = %.3f\n",
                l, length(idx), (fd.x1max - fd.x1min) / (fd.Nx1 ÷ fd.nx_mb[1]) / 2^l,
                median(mx[idx]))
    end
    @printf("  cells: %d = %.1f%% of the uniform mesh;  peak ρ_p/⟨ρ_p⟩ = %.1f\n",
            fd.n_mbs * prod(fd.nx_mb), 100 * fd.n_mbs * prod(fd.nx_mb) /
            ((fd.Nx1 << maximum(lev)) * (fd.Nx2 << maximum(lev))), maximum(mx) / EPS0)
end